In [1]:
# Ô 1 — gói, Java, mốc giờ
import subprocess, sys, os, time
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y openjdk-8-jre-headless >/dev/null 2>&1; java -version")
sh(f"{sys.executable} -m pip install -q 'trl==0.29.1' 'transformers==5.18.0' 'peft==0.21.1' accelerate bitsandbytes pycocoevalcap sacrebleu 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
import importlib.util; assert importlib.util.find_spec("torchao") is None, "DỪNG: torchao còn"
sh(f'{sys.executable} -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"')


openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.8 MB/s eta 0:00:00



5.18.0 0.21.1 2.11.0+cu128 2 Tesla T4



0

In [2]:
# Ô 2 — đường dẫn, md5, hàm chạy song song
import glob, json, shutil, hashlib, re
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
MD5 = {
 "build_branch_data.py": "619e63e123a6dbf60086e65ee94a3912",
 "ex_test_k4.jsonl": "4d8d1646402aeda72f3a826c3d663702",
 "ex_val_k4.jsonl": "3a7ae1e2360659c684234e9d6fbcb866",
 "grpo_spice.py": "07ea87b6d156d1faa391a4274dffd7cf",
 "pred_ck500_test.jsonl": "328847ac96fa3104f76cd997f4091bcd",
 "pred_ck500_vallon.jsonl": "d092faf6f862eec5db71110e6b5963e2",
 "ra_build.py": "7e4ee74274037b83d11db5d63c5e66a8",
 "ra_exemplars.py": "319857d87a6d5fdf8578886ffbb1ac0f",
 "ra_gen.py": "d3969f6105b53b9eae81e7ce521cbb27",
 "ra_score.py": "04aab5e14ee01468b2c6fb8683bc2f8a",
 "ra_train.py": "217c75f188046c4bd279bb11fb1757e4",
 "train_tru_val.jsonl": "edbb231a58574acf6e1968a44d1fa6c6"
}
SRC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ra_train.py", recursive=True)
       if os.path.exists(os.path.join(os.path.dirname(p), "train_tru_val.jsonl"))]
assert len(SRC) == 1, f"DỪNG: cần đúng một dataset ra-sft-script, thấy {SRC}"
SRC = SRC[0]
for f, h in MD5.items():
    assert md5(f"{SRC}/{f}") == h, f"DỪNG: {f} lệch md5"
    if f.endswith(".py"):
        shutil.copy(f"{SRC}/{f}", f"{W}/{f}")
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "p1_train_rows.jsonl" in f)
CK = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
            if md5(p) == "491fa6677340393f1e4464c08a0cec98")
CK = [c for c in CK if not c.strip("/").endswith("ref")]
assert len(CK) == 1, f"DỪNG: không thấy đúng một adapter ck500, thấy {CK}"
CK500 = CK[0]
MERGED = f"{W}/s1_merged"
def chay(jobs, nhip=120):
    ps = []
    for i, (ten, cmd, gpu) in enumerate(jobs):
        if i: time.sleep(90)
        f = open(f"{W}/{ten}.log", "w")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
        ps.append((ten, subprocess.Popen(cmd, cwd=W, stdout=f, stderr=subprocess.STDOUT, env=env), f))
    while any(p.poll() is None for _, p, _ in ps):
        time.sleep(nhip)
        for ten, p, _ in ps:
            L = [l.strip() for l in open(f"{W}/{ten}.log", errors="ignore") if l.strip()]
            print(f" {(time.time() - T_NB) / 3600:5.2f} h · {ten} · {'xong' if p.poll() is not None else 'chạy'} · {L[-1][:110] if L else '..'}", flush=True)
    for ten, p, f in ps:
        f.close()
        print(f"--- {ten} · mã thoát {p.returncode} ---\n" + "".join(open(f"{W}/{ten}.log", errors="ignore").readlines()[-10:]), flush=True)
    return {ten: p.returncode for ten, p, _ in ps}
def hoa():
    r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED], cwd=W, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); assert r.returncode == 0, "DỪNG: hoà S1 lỗi"
print("SRC   ", SRC); print("BUNDLE", BUNDLE); print("CK500 ", CK500, flush=True)


SRC    /kaggle/input/datasets/trangphngngc/ra-sft-script
BUNDLE /kaggle/input/datasets/trangphngngc/fgrb-p1-bundle/fgrb_p1_bundle
CK500  /kaggle/input/datasets/trangphngngc/grpo-spice-ck500


In [3]:
# Ô 3 — hoà S1, dựng dữ liệu
hoa()
DATA = f"{W}/ra_data"
r = subprocess.run(["python", "ra_build.py", "--bundle", BUNDLE, "--kho", f"{SRC}/train_tru_val.jsonl", "--out", DATA, "--chi-train"],
                   cwd=W, capture_output=True, text=True)
print((r.stdout + r.stderr)[-4000:], flush=True); assert r.returncode == 0, "DỪNG: dựng dữ liệu lỗi"
S = json.load(open(f"{DATA}/build_stats.json"))
assert S["kho_buoc"] == 62880, f"DỪNG: kho {S['kho_buoc']}"
assert S["train_rows"] >= 3000, f"DỪNG: chỉ {S['train_rows']} hàng"
assert 80 <= S["train_co_khoi_pct"] <= 91, S
N_STEP = -(-S["train_rows"] // 16)
print(f"✅ dữ liệu: {S['train_rows']} hàng · {N_STEP} bước mỗi nhánh · {(time.time()-T_NB)/3600:.2f} h", flush=True)


[hoà] xong → /kaggle/working/s1_merged · dtype torch.float16



[kho] 62880 bước · 12542 tác vụ · bỏ 346 tác vụ val · 0 tác vụ test chung kho
[train] 3991 hàng · bỏ {'rong': 9} · loại thao tác {'navigate_back': 144, 'click': 2547, 'open_app': 273, 'scroll': 444, 'input_text': 286, 'wait': 289, 'long_press': 7, 'navigate_home': 1}
  train 1000/3991 · 2.0 phút
  train 2000/3991 · 3.7 phút
  train 3000/3991 · 5.4 phút
[xong] {"train_rows": 3991, "train_co_khoi_pct": 85.2, "train_khoi_chua_dich_pct": 12.9, "train_so_vi_du_tb": 4.02, "test_click": 0, "val_click": 0, "val_khoi_chua_dich_pct": 0.0, "kho_buoc": 62880, "bo": {"rong": 9}, "seed": 2026100901, "phut": 7.1}
--- ví dụ câu nhắc RA ---
Mục tiêu: set up an account for me on the FitAI app and enter my details. Gender:female", height: 160", weight: -59kg, age: 45 years goal: keepFit Name- Karin, Experience: Totally newbie, Equipment- Full G
Đã làm: Click on the Continue button at the bottom of the screen → Click on the Keep Fit option → Click on the Continue button at the bottom of the screen
Chữ đọc

✅ dữ liệu: 3991 hàng · 250 bước mỗi nhánh · 0.17 h


In [4]:
# Ô 4a — CHẠY THỬ 2 bước mỗi nhánh (tự dừng notebook nếu lỗi)
LR = 2e-5
jobs = [(f"thu_train_{arm}", ["python", "ra_train.py", "--merged", MERGED, "--ckpt", CK500, "--data", f"{DATA}/train_{arm}.jsonl",
         "--out", f"{W}/thu_{arm}_sft", "--lr", str(LR), "--n", "32"], g) for g, arm in enumerate(("ra", "cont"))]
rc = chay(jobs, nhip=30)
assert all(v == 0 for v in rc.values()), f"DỪNG: chạy thử lỗi {rc}"
SB = 0
for arm in ("ra", "cont"):
    L = open(f"{W}/thu_train_{arm}.log").read()
    print(arm, [l for l in L.splitlines() if "[kiểm nhãn]" in l or "[adapter]" in l or "bước " in l][:5], flush=True)
    SB = max([SB] + [float(m) for m in re.findall(r"([\d.]+) s/bước", L)])
print(f"ƯỚC LƯỢNG lượt thật: {SB:.0f} s/bước × {N_STEP} ≈ {SB * N_STEP / 3600:.1f} h · đã dùng {(time.time()-T_NB)/3600:.2f} h", flush=True)
for p in glob.glob(f"{W}/thu_*_sft"): shutil.rmtree(p, ignore_errors=True)


  0.21 h · thu_train_ra · chạy · [kiểm nhãn] mẫu đầu: 1446 token, tính loss 5 token: 'go back to settings<|im_end|>' · đích 'go back to setting


  0.21 h · thu_train_cont · chạy · [kiểm nhãn] mẫu đầu: 1446 token, tính loss 5 token: 'go back to settings<|im_end|>' · đích 'go back to setting


  0.22 h · thu_train_ra · chạy · bước 1/2 · loss 0.7414 · |g| 2.273 · lr 4.00e-06 · 101.3 s/bước · VRAM 6.6 GiB


  0.22 h · thu_train_cont · chạy · [kiểm nhãn] mẫu đầu: 1446 token, tính loss 5 token: 'go back to settings<|im_end|>' · đích 'go back to setting


  0.22 h · thu_train_ra · chạy · bước 1/2 · loss 0.7414 · |g| 2.273 · lr 4.00e-06 · 101.3 s/bước · VRAM 6.6 GiB


  0.22 h · thu_train_cont · chạy · [kiểm nhãn] mẫu đầu: 1446 token, tính loss 5 token: 'go back to settings<|im_end|>' · đích 'go back to setting


... (9 outputs left out)

--- thu_train_cont · mã thoát 0 ---
[dữ liệu] /kaggle/working/ra_data/train_cont.jsonl · 32 hàng · 2 bước tối ưu × lô 16 · lr 2e-05
[nạp] /kaggle/working/s1_merged · dtype torch.float16 · 4-bit True · Tesla T4
[adapter] /kaggle/input/datasets/trangphngngc/grpo-spice-ck500 · tham số học 14,966,784 (ck500 14,966,784) · dtype tính torch.float16
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[kiểm nhãn] mẫu đầu: 1446 token, tính loss 5 token: 'go back to settings<|im_end|>' · đích 'go back to settings'
  bước 1/2 · loss 0.5506 · |g| 1.851 · lr 4.00e-06 · 89.5 s/bước · VRAM 6.2 GiB
  bước 2/2 · loss 0.3781 · |g| 1.316 · lr 6.00e-06 · 91.9 s/bước · VRAM 6.4 GiB
[xong] 2 bước · 3.1 phút → /kaggle/working/thu_cont_sft/fi

ra ['[dữ liệu] /kaggle/working/ra_data/train_ra.jsonl · 32 hàng · 2 bước tối ưu × lô 16 · lr 2e-05', '[adapter] /kaggle/input/datasets/trangphngngc/grpo-spice-ck500 · tham số học 14,966,784 (ck500 14,966,784) · dtype tính torch.float16', "[kiểm nhãn] mẫu đầu: 1446 token, tính loss 5 token: 'go back to settings<|im_end|>' · đích 'go back to settings'", '  bước 1/2 · loss 0.7414 · |g| 2.273 · lr 4.00e-06 · 101.3 s/bước · VRAM 6.6 GiB', '  bước 2/2 · loss 0.6384 · |g| 3.931 · lr 6.00e-06 · 106.3 s/bước · VRAM 7.1 GiB']


cont ['[dữ liệu] /kaggle/working/ra_data/train_cont.jsonl · 32 hàng · 2 bước tối ưu × lô 16 · lr 2e-05', '[adapter] /kaggle/input/datasets/trangphngngc/grpo-spice-ck500 · tham số học 14,966,784 (ck500 14,966,784) · dtype tính torch.float16', "[kiểm nhãn] mẫu đầu: 1446 token, tính loss 5 token: 'go back to settings<|im_end|>' · đích 'go back to settings'", '  bước 1/2 · loss 0.5506 · |g| 1.851 · lr 4.00e-06 · 89.5 s/bước · VRAM 6.2 GiB', '  bước 2/2 · loss 0.3781 · |g| 1.316 · lr 6.00e-06 · 91.9 s/bước · VRAM 6.4 GiB']


ƯỚC LƯỢNG lượt thật: 106 s/bước × 250 ≈ 7.4 h · đã dùng 0.26 h


In [5]:
# Ô 4b — train thật hai nhánh song song; chừa ~1,6 h cho val + đóng gói
MAXH = max(1.0, 11.0 - 1.6 - (time.time() - T_NB) / 3600)
print(f"max-hours {MAXH:.2f}", flush=True)
jobs = [(f"train_{arm}", ["python", "ra_train.py", "--merged", MERGED, "--ckpt", CK500, "--data", f"{DATA}/train_{arm}.jsonl",
         "--out", f"{W}/{arm}_sft", "--lr", str(LR), "--max-hours", f"{MAXH:.2f}"], g) for g, arm in enumerate(("ra", "cont"))]
rc = chay(jobs)
print(rc, flush=True)
XONG = all(os.path.exists(f"{W}/{arm}_sft/final/adapter_model.safetensors") for arm in ("ra", "cont"))
print("hai nhánh xong final:", XONG, flush=True)


max-hours 9.14


  0.32 h · train_ra · chạy · bước 1/250 · loss 0.7414 · |g| 2.273 · lr 4.00e-06 · 107.9 s/bước · VRAM 6.6 GiB


  0.32 h · train_cont · chạy · bước 1/250 · loss 0.5506 · |g| 1.851 · lr 4.00e-06 · 91.3 s/bước · VRAM 6.2 GiB


  0.35 h · train_ra · chạy · bước 1/250 · loss 0.7414 · |g| 2.273 · lr 4.00e-06 · 107.9 s/bước · VRAM 6.6 GiB


  0.35 h · train_cont · chạy · bước 1/250 · loss 0.5506 · |g| 1.851 · lr 4.00e-06 · 91.3 s/bước · VRAM 6.2 GiB


  0.38 h · train_ra · chạy · bước 1/250 · loss 0.7414 · |g| 2.273 · lr 4.00e-06 · 107.9 s/bước · VRAM 6.6 GiB


... (445 outputs left out)

--- train_ra · mã thoát 0 ---
  bước 210/250 · loss 0.5854 · |g| 1.230 · lr 1.34e-06 · 107.8 s/bước · VRAM 7.3 GiB
  bước 215/250 · loss 0.5296 · |g| 1.410 · lr 1.03e-06 · 107.8 s/bước · VRAM 7.3 GiB
  bước 220/250 · loss 0.4379 · |g| 0.800 · lr 7.61e-07 · 107.7 s/bước · VRAM 7.3 GiB
  bước 225/250 · loss 0.3799 · |g| 1.100 · lr 5.31e-07 · 107.7 s/bước · VRAM 7.3 GiB
  bước 230/250 · loss 0.4143 · |g| 1.165 · lr 3.41e-07 · 107.7 s/bước · VRAM 7.3 GiB
  bước 235/250 · loss 0.6438 · |g| 1.471 · lr 1.92e-07 · 107.8 s/bước · VRAM 7.3 GiB
  bước 240/250 · loss 0.4808 · |g| 1.186 · lr 8.56e-08 · 107.9 s/bước · VRAM 7.3 GiB
  bước 245/250 · loss 0.3770 · |g| 0.728 · lr 2.14e-08 · 108.0 s/bước · VRAM 7.3 GiB
  bước 250/250 · loss 0.4230 · |g| 1.003 · lr 0.00e+00 · 108.1 s/bước · VRAM 7.3 GiB
[xong] 250 bước · 450.3 phút → /kaggle/working/ra_sft/final



--- train_cont · mã thoát 0 ---
  bước 210/250 · loss 0.5966 · |g| 1.033 · lr 1.34e-06 · 96.5 s/bước · VRAM 6.8 GiB
  bước 215/250 · loss 0.5328 · |g| 1.197 · lr 1.03e-06 · 96.5 s/bước · VRAM 6.8 GiB
  bước 220/250 · loss 0.4406 · |g| 0.632 · lr 7.61e-07 · 96.5 s/bước · VRAM 6.8 GiB
  bước 225/250 · loss 0.3656 · |g| 0.960 · lr 5.31e-07 · 96.5 s/bước · VRAM 6.8 GiB
  bước 230/250 · loss 0.4223 · |g| 0.887 · lr 3.41e-07 · 96.5 s/bước · VRAM 6.8 GiB
  bước 235/250 · loss 0.6423 · |g| 1.130 · lr 1.92e-07 · 96.5 s/bước · VRAM 6.8 GiB
  bước 240/250 · loss 0.4826 · |g| 0.930 · lr 8.56e-08 · 96.5 s/bước · VRAM 6.8 GiB
  bước 245/250 · loss 0.3869 · |g| 0.615 · lr 2.14e-08 · 96.5 s/bước · VRAM 6.8 GiB
  bước 250/250 · loss 0.4462 · |g| 0.867 · lr 0.00e+00 · 96.5 s/bước · VRAM 6.8 GiB
[xong] 250 bước · 402.2 phút → /kaggle/working/cont_sft/final



{'train_ra': 0, 'train_cont': 0}


hai nhánh xong final: True


In [6]:
# Ô 5 — val lớn (1.002 click): ra_k4 vs cont vs ck500, 5 thước (không SPICE/BERTScore)
if XONG and (time.time() - T_NB) / 3600 < 10.3:
    vb = ["--merged", MERGED, "--recs", f"{BUNDLE}/p1_val_rows.jsonl", "--ocr", f"{BUNDLE}/ocr.jsonl", "--images", f"{BUNDLE}/images", "--expect", "1002"]
    rc = chay([("val_ra_k4", ["python", "ra_gen.py", *vb, "--ckpt", f"{W}/ra_sft/final", "--ra", f"{SRC}/ex_val_k4.jsonl", "--out", f"{W}/pred_val_ra_k4.jsonl"], 0),
               ("val_cont", ["python", "ra_gen.py", *vb, "--ckpt", f"{W}/cont_sft/final", "--out", f"{W}/pred_val_cont.jsonl"], 1)])
    print(rc, flush=True)
    if all(v == 0 for v in rc.values()):
        r = subprocess.run(["python", "ra_score.py", "--recs", f"{BUNDLE}/p1_val_rows.jsonl", "--pred", f"ck500={SRC}/pred_ck500_vallon.jsonl",
                            "--pred", f"ra_k4={W}/pred_val_ra_k4.jsonl", "--pred", f"cont={W}/pred_val_cont.jsonl",
                            "--ra", f"ra_k4={SRC}/ex_val_k4.jsonl", "--no-bert", "--no-spice", "--out", f"{W}/diem_val_304.json"],
                           cwd=W, capture_output=True, text=True)
        open(f"{W}/diem_val_304.log", "w").write(r.stdout + r.stderr)
        print(r.stdout[-4000:], r.stderr[-1500:], flush=True)
else:
    print("bỏ val (chưa xong train hoặc hết giờ)", flush=True)


  7.84 h · val_ra_k4 · chạy · 50/1002 · 2.5 phút · 3.01 s/bước · 'Click on the save button at the top right corner of the scre'


  7.84 h · val_cont · chạy · warnings.warn(


  7.87 h · val_ra_k4 · chạy · 100/1002 · 5.0 phút · 3.00 s/bước · 'Select the Filter "Vivid".'


  7.87 h · val_cont · chạy · 50/1002 · 2.1 phút · 2.57 s/bước · 'Click on the save button at the top right corner of the scre'


  7.91 h · val_ra_k4 · chạy · 100/1002 · 5.0 phút · 3.00 s/bước · 'Select the Filter "Vivid".'


  7.91 h · val_cont · chạy · 100/1002 · 4.3 phút · 2.56 s/bước · 'Select the Filter Vivid.'


... (42 outputs left out)

--- val_ra_k4 · mã thoát 0 ---
  600/1002 · 29.3 phút · 2.93 s/bước · 'click on the first result'
  650/1002 · 31.8 phút · 2.93 s/bước · 'Click on the General Settings Option'
  700/1002 · 34.2 phút · 2.93 s/bước · 'Click on the Copaya volleyball'
  750/1002 · 36.7 phút · 2.94 s/bước · 'Click on the first image of the flowers.'
  800/1002 · 39.2 phút · 2.94 s/bước · 'click on the APPLY button'
  850/1002 · 41.6 phút · 2.94 s/bước · 'Click on the text bar'
  900/1002 · 43.9 phút · 2.93 s/bước · 'Tap on the picture option'
  950/1002 · 46.3 phút · 2.93 s/bước · 'Swipe down the above circle icon'
  1000/1002 · 48.9 phút · 2.93 s/bước · 'Click on the Entrance Effects option'
[xong] 1002 câu · rỗng 0



--- val_cont · mã thoát 0 ---
  600/1002 · 26.7 phút · 2.67 s/bước · 'click on the first suggested option'
  650/1002 · 28.9 phút · 2.66 s/bước · 'Click on the General Settings'
  700/1002 · 31.1 phút · 2.67 s/bước · 'Click on the copaya volleyball'
  750/1002 · 33.5 phút · 2.68 s/bước · 'Select the first image.'
  800/1002 · 35.7 phút · 2.68 s/bước · 'click on the apply button'
  850/1002 · 37.9 phút · 2.67 s/bước · 'Click on the text bar'
  900/1002 · 40.0 phút · 2.67 s/bước · 'Tap on the picture option'
  950/1002 · 42.2 phút · 2.66 s/bước · 'Tap on the done button at the top left corner of the screen'
  1000/1002 · 44.4 phút · 2.66 s/bước · 'Click on the Entrance Effects option'
[xong] 1002 câu · rỗng 0



{'val_ra_k4': 0, 'val_cont': 0}


[dữ liệu] 1002 bước click · nhánh ['ck500', 'ra_k4', 'cont']
ck500      bleu4 54.81 · meteor 39.25 · rougeL 70.52 · cider_d 448.95 · chrf 64.5 · rong 0 · doi_cau_so_goc_pct 0.0 · trung_chuan_pct 23.1
ra_k4      bleu4 53.95 · meteor 38.47 · rougeL 69.79 · cider_d 459.31 · chrf 63.23 · rong 0 · doi_cau_so_goc_pct 46.1 · trung_chuan_pct 26.6
cont       bleu4 54.35 · meteor 38.72 · rougeL 70.12 · cider_d 453.59 · chrf 63.8 · rong 0 · doi_cau_so_goc_pct 27.0 · trung_chuan_pct 24.6
[chép] ra_k4: chép nguyên văn một ví dụ ở 21.0% bước · trong số đó đúng câu chuẩn 41.0%

| thước | ck500 | ra_k4 | cont |
|---|---:|---:|---:|
| BLEU-4 | 54.81 | 53.95 (-0.86) | 54.35 (-0.46) |
| METEOR | 39.25 | 38.47 (-0.78) | 38.72 (-0.53) |
| ROUGE-L | 70.52 | 69.79 (-0.73) | 70.12 (-0.40) |
| CIDEr-D | 448.95 | 459.31 (+10.36) | 453.59 (+4.64) |
| chrF | 64.50 | 63.23 (-1.27) | 63.80 (-0.70) |
[kết luận] ra_k4: cao hơn ck500 ở 1/5 thước (CIDEr-D) · chưa đạt
[kết luận] cont: cao hơn ck500 ở 1/5 thước (CIDEr-D)

In [ ]:
# Ô 6 — dọn + đóng gói một zip
shutil.rmtree(MERGED, ignore_errors=True)
for p in glob.glob(f"{W}/*.py"): os.remove(p)
shutil.make_archive(f"{W}/ket_qua_304_T", "zip", W, ".")  # zip chứa adapter final/last + log + điểm val
print(sorted(os.listdir(W)), f"{(time.time()-T_NB)/3600:.2f} h", flush=True)
